In [65]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor


# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv("../data/raw/StudentPerformanceFactors.csv")




In [66]:
# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv(
    "../data/raw/StudentPerformanceFactors.csv"
)


# ============================================================
# 2. FEATURE SELECTION
# ============================================================

selected_numeric_columns = [
    "Attendance",
    "Hours_Studied",
    "Previous_Scores"
]


# Get all categorical columns
categorical_columns = (
    df
    .select_dtypes(include="object")
    .columns
    .tolist()
)


# Categorical columns you decided to exclude
excluded_categorical_columns = [
    "School_Type",
    "Gender",
    "Distance_from_Home",
    "Internet_Access",
    "Peer_Influence",
    "Learning_Disabilities"
]


# Keep the remaining categorical columns
selected_categorical_columns = [
    column
    for column in categorical_columns
    if column not in excluded_categorical_columns
]


target_column = "Exam_Score"


# ============================================================
# 3. KEEP ONLY SELECTED COLUMNS
# ============================================================

selected_columns = (
    selected_numeric_columns
    + selected_categorical_columns
    + [target_column]
)

df = df[
    selected_columns
].dropna().copy()


print("Dataset shape:", df.shape)

print("\nSelected numerical columns:")
print(selected_numeric_columns)

print("\nSelected categorical columns:")
print(selected_categorical_columns)


# ============================================================
# 4. X AND y
# ============================================================

X = df.drop(
    columns=target_column
)

y = df[target_column]


# ============================================================
# 5. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# ============================================================
# 6. PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[

        (
            "numeric",
            StandardScaler(),
            selected_numeric_columns
        ),

        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            selected_categorical_columns
        )
    ]
)


# ============================================================
# 7. DEFINE MODELS
# ============================================================

models = {

    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ),

    "CatBoost": CatBoostRegressor(
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1
    )
}


# ============================================================
# 8. HYPERPARAMETER SEARCH SPACES
# ============================================================

param_grids = {

    # --------------------------------------------------------
    # Linear Regression
    # --------------------------------------------------------

    "Linear Regression": {

        "model__fit_intercept": [
            True,
            False
        ]
    },


    # --------------------------------------------------------
    # Random Forest
    # --------------------------------------------------------

    "Random Forest": {

        "model__n_estimators": [
            200,
            300,
            500,
            700
        ],

        "model__max_depth": [
            None,
            10,
            15,
            20,
            30
        ],

        "model__min_samples_split": [
            2,
            5,
            10,
            15
        ],

        "model__min_samples_leaf": [
            1,
            2,
            4,
            8
        ],

        "model__max_features": [
            "sqrt",
            "log2",
            0.5,
            0.7,
            1.0
        ]
    },


    # --------------------------------------------------------
    # XGBoost
    # --------------------------------------------------------

    "XGBoost": {

        "model__n_estimators": [
            300,
            500,
            700,
            1000
        ],

        "model__learning_rate": [
            0.01,
            0.02,
            0.03,
            0.05
        ],

        "model__max_depth": [
            3,
            4,
            5,
            6,
            7
        ],

        "model__min_child_weight": [
            1,
            3,
            5,
            7
        ],

        "model__subsample": [
            0.7,
            0.8,
            0.9,
            1.0
        ],

        "model__colsample_bytree": [
            0.7,
            0.8,
            0.9,
            1.0
        ],

        "model__gamma": [
            0,
            0.1,
            0.3,
            0.5
        ]
    },


    # --------------------------------------------------------
    # CatBoost
    # --------------------------------------------------------

    "CatBoost": {

        "model__iterations": [
            300,
            500,
            700,
            1000
        ],

        "model__learning_rate": [
            0.01,
            0.02,
            0.03,
            0.05
        ],

        "model__depth": [
            4,
            5,
            6,
            7,
            8
        ],

        "model__l2_leaf_reg": [
            1,
            3,
            5,
            7,
            10
        ],

        "model__random_strength": [
            0,
            0.5,
            1,
            2,
            5
        ]
    }
}


# ============================================================
# 9. TRAIN + HYPERPARAMETER TUNING
# ============================================================

results = {}

best_models = {}

best_parameters = {}


for model_name, model in models.items():

    print("\n")
    print("=" * 70)
    print(f"TUNING {model_name}")
    print("=" * 70)


    # --------------------------------------------------------
    # Create pipeline
    # --------------------------------------------------------

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),

            (
                "model",
                model
            )
        ]
    )


    # --------------------------------------------------------
    # Select search method
    # --------------------------------------------------------

    if model_name == "Linear Regression":

        search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grids[model_name],
            scoring="neg_mean_squared_error",
            cv=5,
            n_jobs=-1,
            verbose=1
        )

    else:

        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_grids[model_name],

            # Number of random combinations
            n_iter=20,

            scoring="neg_mean_squared_error",

            # 5-fold cross validation
            cv=5,

            random_state=42,

            n_jobs=-1,

            verbose=1
        )


    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    search.fit(
        X_train,
        y_train
    )


    # --------------------------------------------------------
    # Get best pipeline
    # --------------------------------------------------------

    best_model = search.best_estimator_

    best_models[model_name] = best_model


    # --------------------------------------------------------
    # Save best parameters
    # --------------------------------------------------------

    best_parameters[model_name] = (
        search.best_params_
    )


    # --------------------------------------------------------
    # Best CV RMSE
    # --------------------------------------------------------

    cv_rmse = np.sqrt(
        -search.best_score_
    )


    # --------------------------------------------------------
    # Predict test set
    # --------------------------------------------------------

    y_pred = best_model.predict(
        X_test
    )


    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            y_pred
        )
    )

    r2 = r2_score(
        y_test,
        y_pred
    )


    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    results[model_name] = {

        "MAE": mae,

        "RMSE": rmse,

        "R2": r2,

        "CV_RMSE": cv_rmse
    }


    # --------------------------------------------------------
    # Print results
    # --------------------------------------------------------

    print("\nBest parameters:")
    print(
        search.best_params_
    )

    print(
        f"\nCV RMSE: {cv_rmse:.3f}"
    )

    print("\nTest set results:")

    print(
        f"MAE:  {mae:.3f}"
    )

    print(
        f"RMSE: {rmse:.3f}"
    )

    print(
        f"R2:   {r2:.3f}"
    )


# ============================================================
# 10. FINAL RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    results
).T


print("\n")
print("=" * 70)
print("FINAL MODEL COMPARISON")
print("=" * 70)

display(
    results_df.sort_values(
        by="RMSE"
    )
)


# ============================================================
# 11. BEST MODEL
# ============================================================

best_model_name = (
    results_df[
        "RMSE"
    ]
    .idxmin()
)

print("\nBest model based on test RMSE:")
print(best_model_name)


# ============================================================
# 12. BEST MODEL PARAMETERS
# ============================================================

print("\nBest parameters for the winning model:")

for parameter, value in best_parameters[
    best_model_name
].items():

    print(
        f"{parameter}: {value}"
    )

Dataset shape: (6443, 11)

Selected numerical columns:
['Attendance', 'Hours_Studied', 'Previous_Scores']

Selected categorical columns:
['Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Motivation_Level', 'Family_Income', 'Teacher_Quality', 'Parental_Education_Level']


TUNING Linear Regression
Fitting 5 folds for each of 2 candidates, totalling 10 fits

Best parameters:
{'model__fit_intercept': False}

CV RMSE: 2.324

Test set results:
MAE:  0.905
RMSE: 1.902
R2:   0.753


TUNING Random Forest
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best parameters:
{'model__n_estimators': 200, 'model__min_samples_split': 10, 'model__min_samples_leaf': 4, 'model__max_features': 0.5, 'model__max_depth': None}

CV RMSE: 2.446

Test set results:
MAE:  1.129
RMSE: 2.086
R2:   0.703


TUNING XGBoost
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best parameters:
{'model__subsample': 0.8, 'model__n_estimators': 500, 'model__min_child_weigh

,MAE,RMSE,R2,CV_RMSE
Linear Regression,0.905034,1.902289,0.752848,2.324269
CatBoost,0.955273,1.945691,0.741442,2.348999
XGBoost,1.004005,1.969949,0.734955,2.377471
Random Forest,1.128853,2.085711,0.702889,2.445624



Best model based on test RMSE:
Linear Regression

Best parameters for the winning model:
model__fit_intercept: False


In [72]:
import joblib

final_model = best_models["Linear Regression"]

joblib.dump(
    final_model,
    "../models/student_exam_model.joblib"
)

print("Model saved successfully!")

Model saved successfully!


In [73]:
print(best_models["Linear Regression"].named_steps["preprocessor"].transformers_)

[('numeric', StandardScaler(), ['Attendance', 'Hours_Studied', 'Previous_Scores']), ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Motivation_Level', 'Family_Income', 'Teacher_Quality', 'Parental_Education_Level'])]
